## Calculate emissions & Create CESM-ready files directly

In [1]:
import json
import multiprocessing
import os,shutil,tempfile,time,psutil
import pandas as pd
import xarray as xr
import numpy as np

In [2]:
def GetMW(name):
#
# read CSV with molec. weights for each species
#
    names=pd.read_csv('/glade/u/home/emmons/EMISSIONS/species_molwts.csv')
    names_data=names.to_xarray()
    species=names_data['Species']
    weights=names_data['Mol Wt (g/mol)']
    ind = np.where(species==name,True,False)
    mw = weights[ind].mean()
    print('MW', mw)
    return mw

In [16]:
# Set inputs & outputs
#ef_file = 'G4H_emission_factors.xlsx'
ef_file = '/glade/u/home/emmons/EMISSIONS/HTAP/GFAS/G4H_MOZART_EFs_20260211.csv'
lc_file = '/glade/u/home/emmons/EMISSIONS/HTAP/GFAS/G4H_land_covers.nc'

path_emis = '/glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/gfas4htap/'

# dry matter combustion rate 
dm_file = path_emis+'DM/DM_monthly.nc'

outputs_dir = '/glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/gfas4htap/mozart/01x01/'

#output_species = ['NOx_asNO']
#output_species = ['CO','CO2', 'CH4', 'CO', 'NH3', 'SO2', 'DMS']
#output_species = ['HCN', 'CH3CN',  'C2H2', 'C2H4', 'C2H6', 'C3H6', 'C3H8']
#output_species = ['BIGENE', 'BIGALK', 'ISOP', 'MTERP', 'BENZENE', 'TOLUENE', 'XYLENES']
#output_species = [ 'CH3OH', 'C2H5OH', 'CH2O', 'CH3CHO', 'HCOOH', 'CH3COOH', 'GLYOXAL', 'GLYALD']
output_species = ['CH3COCHO',  'CH3COCH3','HYAC', 'MEK', 'MACR', 'MVK', 'ALKNIT', 'PHENOL', 'CRESOL', 'BZALD', 
                   'BC', 'OC', 'SVOC', 'IVOC', 'SOAE']

#output_species = ['HONO','HNCO', 'C6ALKANES', 'ISOBUTANE', 'NBUTANE', 'IPENTANE', 'NPENTANE', 'STYRENE', 'C2H4O']
#output_species = ['PROPANAL', 'BUTANAL','IPROPANOL', 'FURAN', 'FURAN2M', 'FURAN3M', 'FURAN25M', 'FURANONE', 'FURFAL', 'MALARKY']
#                  'bc_a4', 'pom_a4', 'num_a4_bc', 'num_a4_pom']

time_chunk = 10

In [17]:
for sp1 in output_species:
    # get molecular weight
    if (sp1 == 'NOx_asNO'):
        mw_spec = float(GetMW('NO'))
    else:
        mw_spec = float(GetMW(sp1))

    emi_file = outputs_dir+'gfas4htap-moz_'+sp1+'_2003-2023_01x01.nc'

    print(f'writing emissions of {sp1} to {emi_file}')

    # calculate emissions
    print(f'calculating {sp1} emissions from {dm_file}')

    print(f'reading emission factors from {ef_file}')
    ef_tab = pd.read_csv(ef_file, index_col=0, skiprows=4)

    print(f'reading land cover from {lc_file}')
    lc_map = xr.open_dataset(lc_file)['land_cover']
    lc_def = json.loads(lc_map.attrs['flag_meanings'])
    lc_def.pop('water_ice')

    # create the emission factor map
    ef_map = lc_map.copy()
    # loop over land cover types
    mask = {}
    for lc, lc_val in lc_def.items():
        # identify grid cells of not this land cover type
        mask[lc] = ef_map != lc_val
    for lc, lc_val in lc_def.items():
        # replace values of this land cover type with emission factors
        # while keeping values in grid cells of other land covers
        ef_map = ef_map.where(mask[lc], other=ef_tab[lc][sp1]/1e3)
    ef_map = ef_map.astype('float32')

    dm_ds = xr.open_dataset(dm_file, chunks={'time':time_chunk}, engine='netcdf4')
    if 'DM_budget' in dm_ds.data_vars:
        dm_ds = dm_ds.drop_vars('DM_budget')

    # Emissions = EF * DM burned
    emi_ds1 = ef_map * dm_ds

    # rename dim and vars, swap latitude direction (S to N)
    tmp1 = emi_ds1.rename({'DM':'bb'})
    tmp2 = tmp1.rename( {'latitude':'lat', 'longitude':'lon'} )
    emi_ds_kg = tmp2.reindex(lat = tmp2.lat[::-1])

    # Convert from kg/m2/s to molecules/cm2/s  [molec/mole]/[g/mole]*[g/kg]*[m2/cm2] 
    emi_ds = emi_ds_kg * 6.022E23 / mw_spec * 1.E3 * 1.E-4

    emi_ds['bb'].attrs['long_name'] = ef_tab['long_name'][sp1]
    #emi_ds['bb'].attrs['units'] = 'kg/m2/s'
    emi_ds['bb'].attrs['units'] = 'molecules/cm2/s'
    emi_ds['bb'].attrs['molecular_weight'] = mw_spec
    emi_ds['bb'].attrs['molecular_weight_units'] = 'g mole-1'
    del emi_ds['bb'].attrs['standard_name']
    del emi_ds['bb'].attrs['name']
    del emi_ds['bb'].attrs['flag_values']
    del emi_ds['bb'].attrs['flag_meanings']
    del emi_ds.attrs['standard_name']
    del emi_ds.attrs['long_name']
    del emi_ds.attrs['name']
    del emi_ds.attrs['units']
    del emi_ds.attrs['flag_values']
    del emi_ds.attrs['flag_meanings']

    # Create date
    time_year = emi_ds['time'].values.astype('datetime64[Y]').astype('int') + 1970
    time_month = emi_ds['time'].values.astype('datetime64[M]').astype('int') % 12 + 1
    day_of_each_month = 15
    #time_day = ( emi_ds['time'].values.astype('datetime64[D]') - \
    #                  emi_ds]['time'].values.astype('datetime64[M]') + 1 ).astype('int')
    date_array = []
    for ti, ty in enumerate( time_year ):
        date_array.append( time_year[ti]*10000 + time_month[ti]*100 + day_of_each_month )

    # Add date variable to NetCDF file
    emi_ds['date'] = date_array
    emi_ds['date'].attrs['units'] = 'YYYYMMDD'
    emi_ds['date'].attrs['long_name'] = 'Date'
    
    # write ds to file_name
    # ensure consistent time-chunked compression
    encoding = {}
    for var in emi_ds.data_vars:
        comp = {'zlib': True,
                'complevel': 3}
        if 'time' in emi_ds[var].dims and 'lat' in emi_ds[var].dims and 'lon' in emi_ds[var].dims:
            chunk = min(len(emi_ds['time']), time_chunk)
            comp['chunksizes'] = (chunk, len(emi_ds['lat']), len(emi_ds['lon']))
            emi_ds[var] = emi_ds[var].transpose('time', 'lat', 'lon')
        elif 'time' in emi_ds[var].dims:
            chunk = min(len(emi_ds['time']), time_chunk)
            comp['chunksizes'] = (chunk,)
        encoding[var] = comp

    # set global attributes
    emi_ds.attrs['title'] = "GFAS4HTAP global vegetation fire emissions 2003-2023 for MOZART species"
    emi_ds.attrs['molecular_weight'] = mw_spec
    emi_ds.attrs['molecular_weight_units'] = 'g mole-1'
    emi_ds.attrs['author'] = "Louisa Emmons"
    emi_ds.attrs['institution'] = "NSF NCAR"
    emi_ds.attrs['contact'] = "emmons@ucar.edu"
    emi_ds.attrs['citation'] = "original function from Kaiser et al. (2025) https://doi.org/10.5281/zenodo.13753451"

    # make directory and write file
    out_dir = os.path.split(emi_file)[0]
    os.makedirs(out_dir, exist_ok=True)
    print(f'writing {emi_file}')
    emi_ds.to_netcdf(emi_file, encoding=encoding, mode='w', engine='netcdf4')

    del emi_ds1
    del tmp1
    del tmp2
    del emi_ds_kg


MW <xarray.DataArray 'Mol Wt (g/mol)' ()> Size: 8B
array(72.061)
writing emissions of CH3COCHO to /glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/gfas4htap/mozart/01x01/gfas4htap-moz_CH3COCHO_2003-2023_01x01.nc
calculating CH3COCHO emissions from /glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/gfas4htap/DM/DM_monthly.nc
reading emission factors from /glade/u/home/emmons/EMISSIONS/HTAP/GFAS/G4H_MOZART_EFs_20260211.csv
reading land cover from /glade/u/home/emmons/EMISSIONS/HTAP/GFAS/G4H_land_covers.nc
writing /glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/gfas4htap/mozart/01x01/gfas4htap-moz_CH3COCHO_2003-2023_01x01.nc
MW <xarray.DataArray 'Mol Wt (g/mol)' ()> Size: 8B
array(58.077)
writing emissions of CH3COCH3 to /glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/gfas4htap/mozart/01x01/gfas4htap-moz_CH3COCH3_2003-2023_01x01.nc
calculating CH3COCH3 emissions from /glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/gfas4htap/DM/DM_monthly.nc
reading

In [ ]:
# testing

sp1 = 'CO'
mw_spec = float(GetMW(sp1))
emi_file = outputs_dir+'gfas4htap-moz_'+sp1+'_2003-2023_01x01.nc'

ef_tab = pd.read_csv(ef_file, index_col=0, skiprows=4)
lc_map = xr.open_dataset(lc_file)['land_cover']
lc_def = json.loads(lc_map.attrs['flag_meanings'])
lc_def.pop('water_ice')
# create the emission factor map
ef_map = lc_map.copy()
 # loop over land cover types
mask = {}
for lc, lc_val in lc_def.items():
    # identify grid cells of not this land cover type
    mask[lc] = ef_map != lc_val
for lc, lc_val in lc_def.items():
    # replace values of this land cover type with emission factors
    # while keeping values in grid cells of other land covers
    ef_map = ef_map.where(mask[lc], other=ef_tab[lc][sp1]/1e3)
ef_map = ef_map.astype('float32')
dm_ds = xr.open_dataset(dm_file, chunks={'time':time_chunk}, engine='netcdf4')
if 'DM_budget' in dm_ds.data_vars:
    dm_ds = dm_ds.drop_vars('DM_budget')

emi_ds1 = ef_map * dm_ds
emi_ds1.rename({'DM':'bb'})

emi_ds2 = emi_ds1.rename( {'latitude':'lat', 'longitude':'lon'} )
emi_ds = emi_ds2.reindex(lat = emi_ds2.lat[::-1])


In [ ]:
# Modification of GFAS 'emissions.py'
#  original: Kaiser et al. (2025) https://doi.org/10.5281/zenodo.13753451


In [ ]:
time_chunk = 10

In [ ]:
def to_netcdf(ds, file_name, mode='w', attributes=None):
    """write ds to file_name"""
    # ensure consistent time-chunked compression
    encoding = {}
    for var in ds.data_vars:
        comp = {'zlib': True,
                'complevel': 3}
        if 'time' in ds[var].dims and 'latitude' in ds[var].dims and 'longitude' in ds[var].dims:
            chunk = min(len(ds['time']), time_chunk)
            comp['chunksizes'] = (chunk, len(ds['latitude']), len(ds['longitude']))
            ds[var] = ds[var].transpose('time', 'latitude', 'longitude')
        elif 'time' in ds[var].dims:
            chunk = min(len(ds['time']), time_chunk)
            comp['chunksizes'] = (chunk,)
        encoding[var] = comp

    # set global attributes
    ds.attrs['title'] = "GFAS4HTAP global vegetation fire emissions 2003-2023 for MOZART species"
    ds.attrs['author'] = "Johannes Kaiser, Daniel Holmedal, Martin Ytre-Eide; Louisa Emmons"
    ds.attrs['institution'] = "NILU; NCAR"
    ds.attrs['contact'] = "jkai@nilu.no; emmons@ucar.edu"
    ds.attrs['license'] = "CC-BY-4.0"
    ds.attrs['citation'] = "Kaiser et al. (2025) https://doi.org/10.5281/zenodo.13753451"

    # make directory and write file
    out_dir = os.path.split(file_name)[0]
    os.makedirs(out_dir, exist_ok=True)
    print(f'writing {file_name}')
    ds.to_netcdf(file_name, encoding=encoding, mode=mode, engine='netcdf4')


In [ ]:
def emissions(species, dm_file, emi_file, emission_factor_file, land_cover_file):
    """calculate emissions"""
    print(f'calculating {species} emissions from {dm_file}')

    # print(f'reading emission factors from {emission_factor_file}')
    if emission_factor_file[-3:] == 'csv':
        ef_tab = pd.read_csv(emission_factor_file, index_col=0,skiprows=4)
    elif emission_factor_file[-4:] == 'xlsx':
        ef_tab = pd.read_excel(emission_factor_file, index_col=0, skiprows=4)

    # print(f'reading land cover from {land_cover_file}')
    #land_cover_def_file = land_cover_file.replace('nc4','json')
    #with open(land_cover_def_file) as fp:
    #    lc_def = json.load(fp)
    lc_map = xr.open_dataset(land_cover_file)['land_cover']
    lc_def = json.loads(lc_map.attrs['flag_meanings'])
    lc_def.pop('water_ice')

    # create the emission factor map
    ef_map = lc_map.copy()
    # loop over land cover types
    mask = {}
    for lc, lc_val in lc_def.items():
        # identify grid cells of not this land cover type
        mask[lc] = ef_map != lc_val
    for lc, lc_val in lc_def.items():
        # replace values of this land cover type with emission factors
        # while keeping values in grid cells of other land covers
        ef_map = ef_map.where(mask[lc], other=ef_tab[lc][species]/1e3)
    ef_map = ef_map.astype('float32')

    dm_ds = xr.open_dataset(dm_file, chunks={'time':time_chunk}, engine='netcdf4')
    if 'DM_budget' in dm_ds.data_vars:
        dm_ds = dm_ds.drop_vars('DM_budget')

    emi_ds = ef_map * dm_ds
    emi_ds = emi_ds.rename({'DM':species})
    emi_ds[species].attrs['long_name'] = ef_tab['long_name'][species]
    emi_ds[species].attrs['units'] = 'kg/m2/s'
    del emi_ds[species].attrs['standard_name']
    del emi_ds[species].attrs['name']
    del emi_ds[species].attrs['flag_values']
    del emi_ds[species].attrs['flag_meanings']
    del emi_ds.attrs['standard_name']
    del emi_ds.attrs['long_name']
    del emi_ds.attrs['name']
    del emi_ds.attrs['units']
    del emi_ds.attrs['flag_values']
    del emi_ds.attrs['flag_meanings']

    #print(f'processing {emi_file}')
    to_netcdf(emi_ds, emi_file)


In [ ]:
def budgets(ds_file, species):
    """add the budget variable to ds_file"""
    print(f'calculating {species} budget in {ds_file}')
    shutil.copy(ds_file, ds_file+'_tmp.nc')
    ds = xr.open_dataset(ds_file+'_tmp.nc', engine='netcdf4', chunks={'time':time_chunk})
    R = 6371000.

    # Calculate grid cell areas
    lat_res_rad = np.deg2rad(np.abs(ds.latitude[1] - ds.latitude[0]))
    lon_res_rad = np.deg2rad(np.abs(ds.longitude[1] - ds.longitude[0]))
    equatorial_cell_area = R**2 * lat_res_rad * lon_res_rad
    area = xr.DataArray(equatorial_cell_area * np.cos(np.deg2rad(ds['latitude'])),\
                        dims=('latitude',))

    # Add to the dataset as a new variable
    ds_out = xr.open_dataset(ds_file, mode='a', engine='netcdf4',chunks={'time':time_chunk})
    ds_out[f'{species}_budget'] = (ds[species]*area).sum(dim=['latitude','longitude']).astype('float32')
    ds_out[f'{species}_budget'].attrs['long_name'] = f'global {species} emission rate from vegetation fires'
    ds_out[f'{species}_budget'].attrs['units'] = 'kg/s'
    budget_Tga = float(ds_out[f'{species}_budget'].mean()) / 1e9 * 3600 * 24 * 365.25
    print(f'... {species} has an average global budget of {budget_Tga:.3g} Tg/a')
    to_netcdf(ds_out, ds_file, mode='a')
    os.remove(ds_file+'_tmp.nc')
    return species, budget_Tga, 'Tg/a'

In [ ]:
def emissions_multipro(atuple):
    """wrapper for use with multiprocessing"""
    species, dm_file, emi_file, emission_factor_file, land_cover_file, do_budget = atuple
    with tempfile.NamedTemporaryFile(delete=True) as tmp_file:
        print(f'copy {dm_file} to {tmp_file.name} for processing... ')
        shutil.copyfile(dm_file, tmp_file.name)
        emissions(species, tmp_file.name, emi_file, emission_factor_file, land_cover_file)
        if do_budget:
            budgets(emi_file, species)
    return emi_file

In [ ]:
if __name__ == '__main__':
    """
    main program for GFAS4 htap emissions calculation
    """

    path_emis = '/glade/campaign/acom/acom-weather/emmons/HTAP3/emissions/gfas4htap/'
    
    ### user configuration ###
    # From which dry matter combustion rate shall the emissions be calculated?
    dm_file = path_emis+'DM/DM_monthly.nc'
    # Where should the output go?
    outputs_dir = path_emis+'mozart/'
    # For which species shall emissions be calculated?
    #output_species = ['CO2', 'CH4', 'CO', 'NOx_asNO', 'HONO', 'NH3', 'SO2', 'DMS']
    #output_species = ['HCN', 'CH3CN', 'HNCO', 'C2H2', 'C2H4', 'C2H6', 'C3H6', 'C3H8']
    #output_species = ['BIGENE', 'BIGALK', 'ISOBUTANE', 'NBUTANE', 'IPENTANE', 'NPENTANE']
    #output_species = ['C6ALKANES', 'ISOP', 'MTERP', 'BENZENE', 'TOLUENE', 'XYLENES', 'STYRENE', 'C2H4O']
    #output_species = [ 'CH3OH', 'C2H5OH', 'IPROPANOL', 'CH2O', 'CH3CHO', 'GLYOXAL', 'GLYALD']
    #output_species = ['CH3COCHO', 'PROPANAL', 'BUTANAL', 'CH3COCH3','HYAC', 'MEK', 'MACR', 'MVK']
    #output_species = ['HCOOH', 'CH3COOH', 'FURAN', 'FURAN2M', 'FURAN3M', 'FURAN25M', 'FURANONE', 'FURFAL', 'MALARKY']
    output_species = [ 'ALKNIT', 'PHENOL', 'CRESOL', 'BZALD', 'BC', 'OC', 'SVOC', 'IVOC', 'SOAE']
    #                  'bc_a4', 'pom_a4', 'num_a4_bc', 'num_a4_pom']

    # Store global "budgets", i.e. areal integrations, for each time step, too?
    do_budget = True
    # How many processes may be started in parallel?
    # (Recommendation: Try 1, then use 1 process per 2 CPUs and allow 4GB memory per process.)
    #nproc = min(psutil.cpu_count(logical=False) * 1 // 2, psutil.virtual_memory().total // 1024 ** 3 // 4)
    nproc = 1

    ### GFAS4HTAP static input ###
    #ef_file = 'G4H_emission_factors.xlsx'
    ef_file = '/glade/u/home/emmons/EMISSIONS/HTAP/GFAS/G4H_MOZART_EFs_20260211.csv'
    lc_file = '/glade/u/home/emmons/EMISSIONS/HTAP/GFAS/G4H_land_covers.nc'

    ### get going ###
    start = time.time()

    if nproc == 1: # useful for understanding and debugging
        for s in output_species:
            emi_file = os.path.join(outputs_dir, s, os.path.split(dm_file)[1].replace('DM',s))
            print(f'writing emissions of {s} to {emi_file}')
            emissions(s, dm_file, emi_file, ef_file, lc_file)
            # print(f'calculating budget of {s} ...')
            if do_budget:
                species, budget, units = budgets(emi_file, s)
                print(f'... {species} has a global  budget of {budget:.3g} {units}')
    elif nproc > 1:
        arg_tuple_list = []
        for s in output_species:
            emi_file = os.path.join(outputs_dir, s, os.path.split(dm_file)[1].replace('DM', s))
            arg_tuple_list.append((s, dm_file, emi_file, ef_file, lc_file, do_budget))
        with multiprocessing.Pool(nproc) as pool:
            results = pool.imap(emissions_multipro, arg_tuple_list)
            for result in results:
                print(f'Processed {result}')
    else:
        print(f'invalid parallelisation argument {nproc}')

    # done
    print(f'Execution time: {time.time()-start} seconds')